In [ ]:
import duckdb
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

project_root = Path("..")
cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

con = duckdb.connect()

fuel_cost_per_mile = 0.30


# ------------------------------------------------------------
# 1. Zone-hour trip statistics
# ------------------------------------------------------------

zone_hour_stats = con.execute(f"""
SELECT
    t.PULocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    z.Borough,
    z.Zone,
    COUNT(*) AS trips,

    AVG(total_amount - {fuel_cost_per_mile} * trip_distance)
        AS avg_net_trip_earnings,

    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0
    ) AS avg_trip_minutes

FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet') t

LEFT JOIN read_csv_auto('{raw_dir}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

WHERE
    total_amount > 0
    AND z.Borough != 'EWR'
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19

GROUP BY
    t.PULocationID,
    pickup_hour,
    z.Borough,
    z.Zone

HAVING COUNT(*) >= 100
""").fetchdf()


# ------------------------------------------------------------
# 2. Zone-hour transitions
# ------------------------------------------------------------

zone_hour_transitions = con.execute(f"""
SELECT
    PULocationID,
    DOLocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trips

FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')

WHERE
    total_amount > 0
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19

GROUP BY
    PULocationID,
    DOLocationID,
    pickup_hour
""").fetchdf()


# ------------------------------------------------------------
# 3. Restrict to zones with enough data
# ------------------------------------------------------------

valid_zones = (
    zone_hour_stats.groupby("PULocationID")["trips"]
    .sum()
    .loc[lambda x: x >= 1000]
    .index
)

zone_hour_stats = zone_hour_stats[
    zone_hour_stats["PULocationID"].isin(valid_zones)
]

zone_hour_transitions = zone_hour_transitions[
    zone_hour_transitions["PULocationID"].isin(valid_zones)
]

zone_hour_transitions = zone_hour_transitions[
    zone_hour_transitions["DOLocationID"].isin(valid_zones)
]


zone_hour_transitions["transition_prob"] = (
    zone_hour_transitions["trips"] /
    zone_hour_transitions.groupby(
        ["PULocationID","pickup_hour"]
    )["trips"].transform("sum")
)


# ------------------------------------------------------------
# 4. Improved wait-time proxy using pickups and dropoffs
# ------------------------------------------------------------

zone_hour_flows = con.execute(f"""
SELECT
    zone_id,
    hour,
    SUM(pickups) AS pickups,
    SUM(dropoffs) AS dropoffs
FROM (

    SELECT
        PULocationID AS zone_id,
        EXTRACT(hour FROM tpep_pickup_datetime) AS hour,
        COUNT(*) AS pickups,
        0 AS dropoffs
    FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19
    GROUP BY
        PULocationID,
        hour

    UNION ALL

    SELECT
        DOLocationID AS zone_id,
        EXTRACT(hour FROM tpep_pickup_datetime) AS hour,
        0 AS pickups,
        COUNT(*) AS dropoffs
    FROM read_parquet('{cleaned_dir}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19
    GROUP BY
        DOLocationID,
        hour

)
GROUP BY
    zone_id,
    hour
""").fetchdf()

zone_hour_flows = zone_hour_flows[
    zone_hour_flows["zone_id"].isin(valid_zones)
].copy()

# avoid divide-by-zero
zone_hour_flows["pickups"] = zone_hour_flows["pickups"].clip(lower=1)

# more dropoffs relative to pickups means more taxi competition
zone_hour_flows["dropoff_pickup_ratio"] = (
    zone_hour_flows["dropoffs"] / zone_hour_flows["pickups"]
)

# scaling constant in minutes
wait_scale = 5.0

zone_hour_flows["expected_wait_minutes"] = (
    wait_scale * zone_hour_flows["dropoff_pickup_ratio"]
)

# keep values in a realistic range
zone_hour_flows["expected_wait_minutes"] = (
    zone_hour_flows["expected_wait_minutes"].clip(lower=1, upper=30)
)

zone_hour_flows.head()


# ------------------------------------------------------------
# 5. Build lookup dictionaries
# ------------------------------------------------------------

stats_lookup = {}
for _, row in zone_hour_stats.iterrows():
    stats_lookup[(int(row["PULocationID"]), int(row["pickup_hour"]))] = {
        "avg_net_trip_earnings": float(row["avg_net_trip_earnings"]),
        "avg_trip_minutes": float(row["avg_trip_minutes"]),
        "zone_name": row["Zone"],
        "borough": row["Borough"],
    }

transitions_lookup = {}
for (pu, hr), group in zone_hour_transitions.groupby(["PULocationID", "pickup_hour"]):
    destinations = group["DOLocationID"].astype(int).to_numpy()
    probabilities = group["transition_prob"].to_numpy(dtype=float)
    probabilities = probabilities / probabilities.sum()

    transitions_lookup[(int(pu), int(hr))] = {
        "destinations": destinations,
        "probabilities": probabilities,
    }

wait_lookup = {}
for _, row in zone_hour_flows.iterrows():
    wait_lookup[(int(row["zone_id"]), int(row["hour"]))] = float(row["expected_wait_minutes"])

print("stats keys:", len(stats_lookup))
print("transition keys:", len(transitions_lookup))
print("wait keys:", len(wait_lookup))



# ------------------------------------------------------------
# 6. Simulation
# ------------------------------------------------------------

def simulate_shift(start_zone, start_hour=8, shift_minutes=480, rng=None):

    if rng is None:
        rng = np.random.default_rng()

    current_zone = int(start_zone)

    elapsed_minutes = 0
    total_net_earnings = 0
    trip_count = 0

    while elapsed_minutes < shift_minutes:

        current_hour = start_hour + int(elapsed_minutes / 60)

        if current_hour > 19:
            break

        stats_key = (current_zone, current_hour)

        trans_key = (current_zone, current_hour)

        if stats_key not in stats_lookup:
            break

        if trans_key not in transitions_lookup:
            break

        trip_stats = stats_lookup[stats_key]

        trip_minutes = trip_stats["avg_trip_minutes"]

        wait_minutes = wait_lookup.get(
            (current_zone, current_hour),
            5
        )

        total_net_earnings += trip_stats["avg_net_trip_earnings"]

        elapsed_minutes += trip_minutes + wait_minutes

        trip_count += 1

        destinations = transitions_lookup[trans_key]["destinations"]

        probabilities = transitions_lookup[trans_key]["probabilities"]

        current_zone = int(
            rng.choice(destinations, p=probabilities)
        )


    hours_worked = elapsed_minutes / 60

    return {

        "net_earnings_per_hour":
        total_net_earnings / hours_worked,

        "trip_count": trip_count
    }


def evaluate_start_zone(start_zone, start_hour=8, n_simulations=200):

    results = []

    rng = np.random.default_rng(123)

    for _ in range(n_simulations):

        sim = simulate_shift(
            start_zone=start_zone,
            start_hour=start_hour,
            rng=rng
        )

        results.append(sim["net_earnings_per_hour"])

    return np.nanmean(results), np.nanstd(results)



# ------------------------------------------------------------
# 7. Evaluate zones
# ------------------------------------------------------------

zone_results = []

for zone_id in sorted(valid_zones):

    mean_eph, std_eph = evaluate_start_zone(zone_id)

    zone_name = zone_hour_stats.loc[
        zone_hour_stats.PULocationID == zone_id,
        "Zone"
    ].iloc[0]

    borough = zone_hour_stats.loc[
        zone_hour_stats.PULocationID == zone_id,
        "Borough"
    ].iloc[0]

    zone_results.append({

        "PULocationID": zone_id,
        "Zone": zone_name,
        "Borough": borough,
        "expected_net_earnings_per_hour": mean_eph,
        "std_net_earnings_per_hour": std_eph
    })


zone_results = pd.DataFrame(zone_results)

zone_results = zone_results.sort_values(
    "expected_net_earnings_per_hour",
    ascending=False
)

zone_results.head()



# ------------------------------------------------------------
# 8. Bar chart of best zones
# ------------------------------------------------------------

top_zones = zone_results.head(15)

plt.figure(figsize=(10,6))

plt.barh(
    top_zones.Zone,
    top_zones.expected_net_earnings_per_hour
)

plt.gca().invert_yaxis()

plt.xlabel("Expected Net Earnings per Hour")

plt.title("Best Starting Zones (8AM Shift)")

plt.show()



# ------------------------------------------------------------
# 9. Map visualization
# ------------------------------------------------------------

zones_map = gpd.read_file(
    project_root / "data/raw/taxi_zones/taxi_zones.shp"
)

map_data = zones_map.merge(

    zone_results,

    left_on="LocationID",

    right_on="PULocationID",

    how="left"
)


fig, ax = plt.subplots(1,1, figsize=(12,10))

map_data.plot(

    column="expected_net_earnings_per_hour",

    cmap="plasma",

    linewidth=0.5,

    edgecolor="black",

    legend=True,

    ax=ax
)

ax.set_title(
    "Expected Net Earnings per Hour by Starting Zone\n(8-Hour Shift Starting at 8 AM)"
)

ax.axis("off")

plt.show()

In [ ]:
print(zone_hour_flows["expected_wait_minutes"].describe())
print(zone_hour_flows["expected_wait_minutes"].min())
print(zone_hour_flows["expected_wait_minutes"].head())